# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [1]:
%load_ext autoreload
%autoreload 2

import json
from _campaign_lib import *

# --- Services ---
svc = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
campaign_config = {
    "sample_size": 15,              # queries per eval step (0 = all)
    "exploration_sample_size": 10,  # queries per scan/grid point (can be smaller)
    "exploration_rate": 0.5,        # PRIMARY KNOB: 0.0=conservative, 1.0=aggressive
    "improvement_areas": "profile schema quality, web search relevance",
    "exclude_steps": ["llm_ranking"],    # steps to skip (e.g. ["entity_profiling"])
    "pipeline_overrides": {},
    "optimization": {
        "patience": 2,                   # default: 3
        "max_rounds": None,              # default: 10 (None = unlimited)
        "degradation_threshold": 0.4,    # fraction of degraded queries to trigger escalation
        "backend_warning_threshold": 2,  # degradation resets before backend advisory
        "enable_l2": True,               # L2 refine_context on escalation
        "enable_l3": True,               # L3 modify_plan on L2 stall
        "l2_patience": None,             # default: 2
        "l3_patience": None,             # default: 1
    },
    "eval_llm": {
        "model":       "moonshotai/kimi-k2-instruct-0905",
        "provider":    "groq",
        "temperature": 0.4,
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",
        # "model": "claude-sonnet-4-6",
        # "model": "claude-haiku-4-5-20251001",
        "max_tokens": 2000,
    },
    "pipeline_params": None        # set by configure_pipeline()
}

# --- Pipeline snapshot & params ---
pipeline_config_full = await show_pipeline_snapshot(svc)
pipeline_params = configure_pipeline(svc, campaign_config)

Backend: http://127.0.0.1:8000


2026-03-24 13:45:03 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"
2026-03-24 13:45:03 INFO     [api.services.pipeline_discovery] Parsed pipeline 'TermNorm' with 6 steps
2026-03-24 13:45:03 INFO     [api.services.campaign.campaign_init] Pipeline schema loaded: termnorm vv1.1
2026-03-24 13:45:03 INFO     [httpx] HTTP Request: GET http://127.0.0.1:8000/pipeline "HTTP/1.1 200 OK"


Pipeline: termnorm (6 steps)
Experiment: production_historical (40 queries, 93 session terms)
Experiment : production_historical
Mappings   : 887 total, 812 with verified ground truth
Queries    : 40  |  Session terms: 93
Loaded task description: 3751 chars from LCA_INPUT_PATTERNS.md
  PIPELINE SNAPSHOT: TermNorm v1.1
  Nodes:   ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking', 'direct_prompt']
  Schemas: ['entity_profile/1', 'llm_ranking_output/1']
  Prompts: ['entity_profiling/1', 'llm_ranking/1']

{
  "name": "TermNorm",
  "version": "v1.1",
  "description": "TermNorm AI terminology normalization pipeline",
  "required_step": "entity_profile",
  "template_variables": [
    "{{core_concept}}",
    "{{entity_profile_json}}",
    "{{matches}}"
  ],
  "dataset_name": "termnorm_ground_truth",
  "available_models": [
    "moonshotai/kimi-k2-instruct-0905",
    "meta-llama/llama-4-scout-17b-16e-instruct",
    "moonshotai/kimi-k2-instruct"

In [2]:
#@title Task context decomposition
from api.services.llm_client import get_llm_client

task_context = await decompose_task_context(
    TASK_DESCRIPTION, campaign_config, svc,
    llm_client=get_llm_client("groq"),
    model="moonshotai/kimi-k2-instruct-0905",
)

2026-03-24 13:45:05 WARNING  [langfuse] Prompt 'optimizer_restructure-label:production' not found during refresh, evicting from cache.


TASK CONTEXT DECOMPOSITION
  domain: Life Cycle Assessment
  pipeline_purpose: TermNorm normalizes messy user material strings into canonical LCA-database identifiers so downstream LCA calculations use correct emission factors.
  data_characteristics: Single-line strings (10-120 chars) mixing alloy codes, brand names, standards, geography tags; multilingual; ~50 k unique inputs/day.
  optimization_goals: Top-1 exact-match accuracy ≥ 96 %; minimize false positives on no-match inputs; latency < 80 ms.
  key_challenges: Brand→polymer mapping requires external knowledge; geography tags select among 5-30 variants; compositional blends may need decomposition; standards encode identity indirectly; no-match cases must return '--'.

  Consultation: Focus first on enriching the profile schema: add a small curated dictionary (brand→polymer, alloy code→base metal) and a geography-tag regex so the LLM can strip/select variants without hallucinating. For web search relevance, embed the canonical dat

In [3]:
#@title Load datasets & session terms
train_data, session_terms = prepare_datasets(
    svc["store"], svc.get("backend_id", ""),
    excel_path=r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx",
)
svc["session_terms"] = session_terms


  Train              : 984 queries
  Test (processes)   : 82 queries
  Test (material)    : 165 queries
  ------------------------------------------------
  Combined queries   : 820 (deduplicated)
  Session identifiers: 94 unique targets


In [4]:
#@title Prepare evaluation context
campaign_rounds = []
baseline_results = []
baseline_ps, eval_data, backend_status = await prepare_eval_context(svc, train_data)



BACKEND STATUS
  Session Active                 True
  Active Sessions                1
  Terms Loaded                   94
  Match Database Identifiers     111
  Match Database Aliases         722
  Experiments Count              4
  Mappings Count                 1126
  Pipeline Version               v1.1
  Llm Provider                   groq
  Llm Model                      moonshotai/kimi-k2-instruct-0905
  ------------------------------------------------
  Experiments                   
    0_production_realtime        0 mappings
    1_production_historical      887 mappings
    2_bom_materials              159 mappings
    3_bom_processing             80 mappings

Evaluation data: 984 queries


In [5]:
#@title Run baseline evaluation (optional)
RUN_BASELINE = False  # Set True to evaluate baseline before exploration
if RUN_BASELINE:
    campaign_rounds, baseline_results = await run_baseline_eval(
        baseline_ps, eval_data, campaign_config, svc,
    )


In [6]:
#@title Experiment dashboard
EXPERIMENT_ID = None  # Set to hex ID to resume (e.g. '68e2c5')

if EXPERIMENT_ID:
    pipeline_params = load_and_apply_experiment(
        svc, campaign_config, EXPERIMENT_ID, pipeline_params,
    )

show_experiment_dashboard(
    svc=svc, experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config, eval_data=eval_data,
    pipeline_params=locals().get("pipeline_params"),
    baseline_prompt_state=campaign_rounds[0]["prompt_state"].model_dump() if campaign_rounds else None,
)



  EXPERIMENT DASHBOARD (termnorm-local)
  Dataset runs: 50 total (2 77e7e77777e7, 48 other)
  Best result: 66.7% (scan)

  No campaigns yet.
  Set experiment_id="<short_id>" to see full config and diff
  Active: cycle_26439cac667c



[]

## 3. Explore

Exploration via **Smart Search** (scan advisor + sensitivity scan).

In [7]:
#@title 3a. Smart Search — Browse variant library
# show_variant_library()
# show_variant_library(source="PromptWizard")
# show_variant_library(axes=["thinking_style", "persona"])

# preview_advisor_prompt()
preview_advisor_prompt(campaign_config, svc, task_description=task_context, raw=True)


2026-03-24 13:45:10 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


You are an expert prompt optimization advisor. Recommend which axes (parameters and prompt fields) to prioritize in a sensitivity scan.

## Constraints (apply strictly)
- Do NOT recommend *_model axes — place them in axes_to_skip.
- Response must fit within 1500 tokens. Be terse.

## Pipeline: TermNorm AI terminology normalization pipeline
Steps execute sequentially — each step's output feeds the next:
[
  {
    "name": "cache_lookup",
    "node_role": "cache",
    "short_circuit": true
  },
  {
    "name": "fuzzy_matching",
    "node_role": "candidate_source",
    "short_circuit": true
  },
  {
    "name": "web_search",
    "node_role": "enricher"
  },
  {
    "name": "entity_profiling",
    "node_role": "enricher"
  },
  {
    "name": "token_matching",
    "node_role": "candidate_source"
  }
]

## Task Context
- **domain**: Life Cycle Assessment
- **pipeline_purpose**: TermNorm normalizes messy user material strings into canonical LCA-database identifiers so downstream LCA calculatio

In [8]:
#@title Scan advisor
advisory, scan_variants, schema_labels = await run_scan_advisor(
    campaign_config, svc,
    task_description=locals().get("task_context") or locals().get("TASK_DESCRIPTION", ""),
)

2026-03-24 13:45:10 INFO     [api.services.search.smart_search] filter_variant_library: dropped all prompt_fields (llm_ranking not active)


SCAN ADVISOR -- pipeline-aware sensitivity setup
  Pipeline: termnorm (v1.1)
  Steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching', 'llm_ranking']
  Excluded: ['llm_ranking']
  Task context: Life Cycle Assessment — TermNorm normalizes messy user material strings into canonic
  Calling moonshotai/kimi-k2-instruct-0905 ...



2026-03-24 13:45:16 WARNING  [api.services.search.scan_advisor] Scan advisor validation: prompt_field axis 'profiling_schema' not found in variant_library prompt_fields: []


----------------------------------------------------------------------
PRIORITY AXES (ranked by importance)
----------------------------------------------------------------------
  1. [HIGH] query_prefix (pipeline_param) -- step: web_search
     Shapes search space for brand decoding
     Values: ['LCA database', 'material', 'polymer', 'alloy']
  2. [HIGH] query_suffix (pipeline_param) -- step: web_search
     Adds geography or standard context
     Values: ['ecoinvent', 'GaBi', 'RER', 'GLO']
  3. [MEDIUM] num_results (pipeline_param) -- step: web_search
     More hits raise recall for rare brands
     Values: [10, 30, 50]
  4. [HIGH] fuzzy_threshold (pipeline_param) -- step: fuzzy_matching
     Tight threshold cuts false positives
     Values: [75, 85, 95]
  5. [MEDIUM] max_token_candidates (pipeline_param) -- step: token_matching
     More candidates catch compositional blends
     Values: [10, 30, 50]
  6. [HIGH] profiling_schema (prompt_field)
     Add geography_alias for location 

In [ ]:
#@title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 3  # queries per scan variant (0 = use all)

scan_variants = {
    # ── Token matching ───────────────────────────────────────────────────
    'max_token_candidates': [10, 30, 50],
    # ── Web search: query framing ────────────────────────────────────────
    'query_prefix': [
        # --- Database-oriented ---
        'ecoinvent', 'GaBi', 'ecoinvent LCA database material name',
        'material composition LCA', # 'ecoinvent equivalent', 'ecoinvent match for', 'identify LCA material for',
        'what is',# 'define', 'identify', 'describe material',
        # 'chemical composition of', 'CAS number', 'IUPAC name for',
        'manufacturer datasheet', 'technical data sheet', 'product specification', # 'material safety data sheet',
        'manufactured from', 'production process for', # 'raw material for',
        # --- Synonym / translation ---
        'also known as', # 'synonym for', 'equivalent material', 'alternative name for',
        'wikipedia', # 'material properties of',
    ],
    # ── Web search: volume knobs ─────────────────────────────────────────
    'max_sites': [3, 7, 12],
    'num_results': [5, 20, 40],
    'content_char_limit': [400, 800, 1500],
    # ── Fuzzy matching ───────────────────────────────────────────────────
    # 'fuzzy_threshold': [50, 70, 90],
    # 'fuzzy_scorer': ['ratio', 'WRatio', 'token_set_ratio'],
    # ── Entity profiling: LLM tuning ─────────────────────────────────────
    'profiling_temperature': [0.0, 0.3, 0.7],
    'raw_content_limit': [1000, 2500, 8000],
    # ── Entity profiling: schema mutations ───────────────────────────────
    'profiling_schema': [
        # --- LCA-database-specific (original) ---
        [['+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'],
         ['+', 'database_format_hint', 'string', False, "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'"]],
        [['+', 'lca_database_names', 'array', True, "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'"]],
        [['-', 'manufacturing_processes'], ['-', 'applications'],
         ['+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions']],
        # [['~', 'notes', 'material_category', 'string', True, "The broad LCA material category this entity belongs to, e.g. 'polyethylene', 'brass', 'steel'"]],
        [['-', 'applications'], ['+', 'ecoinvent_candidate_names', 'array', True, 'Exact ecoinvent activity names this entity most likely maps to']],
        [['-', 'manufacturing_processes'], ['+', 'database_search_tokens', 'array', True, 'Optimized search tokens for LCA database lookup including spelling variants']],
        # [['~', 'classification_aliases', 'classification_aliases', 'array', True, 'All valid ecoinvent/GaBi naming variants including geography codes and system model suffixes']],
        # --- Minimalist: strip to core matching signals ---
        [['-', 'applications'], ['-', 'manufacturing_processes'], ['-', 'notes'], ['-', 'technical_specifications']],
        # --- Chemical identity ---
        [['+', 'cas_number', 'string', False, 'CAS registry number if identifiable from context'],
         ['+', 'chemical_formula', 'string', False, 'Chemical formula or molecular structure notation']],
        # --- Trade name decoding ---
        [['~', 'key_properties', 'trade_names', 'array', True, 'Known commercial/trade names and brand names for this material, e.g. Makrolon=polycarbonate, Delrin=POM']],
        # --- Material hierarchy (specific→generic) ---
        [['+', 'material_hierarchy', 'array', False, 'Classification chain from specific to generic, e.g. [Makrolon 2805, polycarbonate, thermoplastic, polymer]']],
        # --- Process-centric (flip perspective from material to process) ---
        [['~', 'applications', 'production_route', 'string', False, 'Primary production/manufacturing route e.g. injection molding, extrusion, casting'],
         ['~', 'notes', 'form_factor', 'string', False, 'Physical form: granulate, sheet, rod, wire, powder, liquid, film']],
        # --- Standards-focused ---
        [['+', 'applicable_standards', 'array', False, 'DIN/ISO/EN/ASTM standards that reference or define this material'],
         ['-', 'applications']],
        # # --- Geography-aware ---
        # [['+', 'supply_chain_geography', 'string', False, 'Most likely geographic origin or market region for this material']],
        # # --- Confidence / ambiguity signal ---
        # [['+', 'confidence_level', 'string', False, 'How confident the model is in the identification: high/medium/low/ambiguous'],
        #  ['+', 'ambiguity_notes', 'string', False, 'What makes this input hard to identify — abbreviation, trade name, multi-material, etc.']],
        # # --- Spelling / language variant boost ---
        # [['+', 'spelling_variants', 'array', False, 'All known spelling variants across EN/DE/FR, e.g. aluminium/aluminum, polyamid/polyamide'],
        #  ['-', 'notes']],
        # --- Werkstoff / alloy code decoding ---
        [['+', 'material_code_decoded', 'string', False, 'Decoded meaning of any material code, Werkstoff number, or alloy designation present in the input'],
         ['+', 'base_material', 'string', False, 'The fundamental base material, e.g. brass, steel, polycarbonate']],
        # --- Functional equivalence ---
        [['~', 'applications', 'functional_unit', 'string', False, 'The functional unit this material serves, e.g. structural plastic, electrical insulation, food-grade packaging'],
         ['+', 'substitutes', 'array', False, 'Materials that could serve the same functional role']],
    ],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, svc=svc)

  max_token_candidates: [10, 30, 50]
  query_prefix: ['ecoinvent', 'GaBi', 'ecoinvent LCA database material name', 'material composition LCA', 'what is', 'manufacturer datasheet', 'technical data sheet', 'product specification', 'manufactured from', 'production process for', 'also known as', 'wikipedia']
  max_sites: [3, 7, 12]
  num_results: [5, 20, 40]
  content_char_limit: [400, 800, 1500]
  profiling_temperature: [0.0, 0.3, 0.7]
  raw_content_limit: [1000, 2500, 8000]
  profiling_schema: (baseline + 13 mutations)
    [0] (baseline)
    [1] ('+', 'geography_scope', 'array', False, 'Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA'), ('+', 'database_format_hint', 'string', False, 'Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'')
    [2] ('+', 'lca_database_names', 'array', True, 'Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming 

In [10]:
#@title Prepare scan baseline
# Scan always uses fresh pipeline defaults (not experiment overrides) so the
# baseline content hash matches previous runs regardless of EXPERIMENT_ID.
scan_pipeline_params = configure_pipeline(svc, campaign_config)
scan_baseline_sp, scan_coverage = await prepare_scan_baseline(
    baseline_ps, campaign_config,
    pipeline_params=scan_pipeline_params,
    svc=svc, scan_variants=scan_variants,
)

2026-03-24 13:45:16 INFO     [api.services.search.context] restructure_context_cached: hit (alias group)


Active steps: ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching']
  Excluded: ['llm_ranking']
  Restructured baseline fields (cached):
    persona: You are a candidate evaluation expert.
    task_intent: Rank 20 entity-name candidates by how well they match a target profile and its c...
    problem_description: Given a JSON entity profile and a core concept, score and rank candidate names o...
    instruction: 1) Extract the entity_category and key distinguishing features from the profile....
    thinking_style: Think step-by-step: isolate distinguishing features → compare each candidate → a...
    answer_format: Valid JSON only: { "reasoning": "...", "ranked_candidates": [ { "rank": 1, "cand...
  Search baseline: 402159cb8d0d (render: 868 chars)


2026-03-24 13:45:16 INFO     [api.services.search.coverage] build_prompt_result_index: 50 runs -> 3 unique prompts, 26 total query results



  Historical data: 26 results across 3 unique prompts
  Matching runs (sp_hash): 49, 13 cached results

  Scan variant coverage (49 matching runs):
    max_token_candidates     10→2 ✓  30→2 ✓  50→2 ✓
    query_prefix             12/12 values tested
    max_sites                3→1 ✓  7→1 ✓  12→1 ✓
    num_results              5→1 ✓  20→1 ✓  40→1 ✓
    content_char_limit       400→1 ✓  800→1 ✓  1500→1 ✓
    profiling_temperature    0.0→1 ✓  0.3→1 ✓  0.7→1 ✓
    raw_content_limit        1000→1 ✓  2500→1 ✓  8000→1 ✓
    profiling_schema         14/14 values tested


In [11]:
#@title Sensitivity scan
scan_df, axis_profiles = await sensitivity_scan(
    scan_baseline_sp, scan_variants, eval_data,
    sample_size=locals().get('scan_sample_size', 10),
    svc=svc, experiment_id=EXPERIMENT_ID or "",
)

Running sensitivity scan...

  Baseline field values:
    persona: You are a candidate evaluation expert.
    task_intent: Rank 20 entity-name candidates by how well they match a target profile and its c...
    problem_description: Given a JSON entity profile and a core concept, score and rank candidate names o...
    instruction: 1) Extract the entity_category and key distinguishing features from the profile....
    thinking_style: Think step-by-step: isolate distinguishing features → compare each candidate → a...
    answer_format: Valid JSON only: { "reasoning": "...", "ranked_candidates": [ { "rank": 1, "cand...

  Axes: 8, variants: 44, queries/variant: 3, cached results: 180
  Estimated calls: ~132
  Evaluating baseline...
        MISS 3/20  [token] 📖  PA66-GF25 ULTRAMID A3UG5 RAL7035 grey          -> Glass fibre reinforced plastic | 50 14.5s
                    ⚠ web_search: 7 of 15 fetched URLs returned content (8 filtered: 6×skip_extension, 1×too_short, 1×http_403)
        MIS

In [12]:
# #@title Scan analytics (uncomment to display)
# if scan_df is not None and not scan_df.empty:
#     show_scan_leaderboard(scan_df, axis_profiles)
#     difficulty_df = show_scan_query_difficulty(svc["store"], svc["backend_id"])

In [13]:
#@title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df, axis_profiles, scan_baseline_sp, scan_variants,
    campaign_rounds, campaign_config,
)

2026-03-24 13:45:17 INFO     [api.services.search.smart_search] select_scan_winner: 0 prompt changes, 2 param changes from 2 improving axes


Selected best from 2 improving axes:
  query_prefix              best_delta=+30.0%  value_idx=3  acc=66.7%
  profiling_temperature     best_delta=+30.0%  value_idx=0  acc=66.7%
Pipeline params updated: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'query_prefix': 'material composition LCA', 'profiling_temperature': 0.0}
Updated pipeline_params: {'steps': ['cache_lookup', 'fuzzy_matching', 'web_search', 'entity_profiling', 'token_matching'], 'query_prefix': 'material composition LCA', 'profiling_temperature': 0.0}

Round    Accuracy   Rolling Avg    Trend
  search    33.3%        33.3%  -


## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [14]:
#@title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds, eval_data, campaign_config,
    pipeline_params=pipeline_params,
    scan_df=locals().get("scan_df"),
    axis_profiles=locals().get("axis_profiles"),
    scan_variants=locals().get("scan_variants"),
    difficulty_df=locals().get("difficulty_df"),
)


  FEEDBACK CYCLE PRE-FLIGHT
  Baseline accuracy      : 33.3%
  Baseline prompt        : 1) Extract the entity_category and key distinguishing features from the profile....
  ------------------------------------------------------------------
  Max rounds             : unlimited
  Candidates per round   : 5
  Queries per eval       : 15 of 984
  Improvement threshold  : 1.0%
  Patience (L1)          : 2 rounds
  L2 (refine context)    : enabled, patience=2
  L3 (modify plan)       : enabled, patience=1
  ------------------------------------------------------------------
  Candidate model        : moonshotai/kimi-k2-instruct-0905
  Creativity             : 0.7
  Pipeline               : 5 of 6 steps
    Steps                : cache_lookup, fuzzy_matching, web_search, entity_profiling, token_matching
    Excluded             : llm_ranking
  Strategy               : SCAN-AWARE

  ROUND PIPELINE (what happens each round)
  ------------------------------------------------------------------
 

In [15]:
#@title Run optimization (feedback cycle)
# Force-reload api modules (ensures code edits take effect without kernel restart)
import importlib, sys
for _m in [
    "api.services.campaign.escalation",
    "api.services.campaign.layer_transitions",
    "api.services.campaign.critique",
    "api.services.campaign.models",
    "api.services.prompt_optimizer",
    "api.services.campaign.feedback_cycle",
]:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])

campaign_rounds = await run_feedback_cycle_notebook(
    campaign_rounds, eval_data, campaign_config,
    svc=svc,
    pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=locals().get("EXPERIMENT_ID"),
    task_context=locals().get("task_context"),
)

  Interrupt of cells can take up to 60 seconds!
  If a dialog pops up, click 'Cancel' and wait 20 seconds.

╔════════════════════════════════════════════════════════════════════╗
║  FEEDBACK CYCLE STARTING                                           ║
╠════════════════════════════════════════════════════════════════════╣
║  Baseline       33.3%                                              ║
║  Max rounds     999            Patience    2                       ║
║  Candidates     5                                                  ║


2026-03-24 13:45:45 INFO     [api.services.campaign.feedback_cycle] Using provided baseline (acc=0.333)
2026-03-24 13:45:45 INFO     [api.services.campaign.feedback_cycle] Cycle identity: cycle_dbb05e948b53
2026-03-24 13:45:45 WARNING  [api.services.campaign.feedback_cycle] Cycle resume setup failed — running fresh
Traceback (most recent call last):
  File "C:\Users\dsacc\Desktop\PromptPotter\prompt-potter-optimizer\api\services\campaign\feedback_cycle.py", line 306, in _resume_or_create_campaign
    campaign_store.create(config.backend_id, cycle_id, {
    ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        "type": "feedback_cycle",
        ^^^^^^^^^^^^^^^^^^^^^^^^^
        "config": config.model_dump(),
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        "baseline_accuracy": baseline_accuracy,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    })
    ^^
  File "C:\Users\dsacc\Desktop\PromptPotter\prompt-potter-optimizer\api\services\stores\campaign_store.py", line 78, in create
 

║  Sample size    15 of 984                                          ║
║  Min detectable ±36.2% (α=0.05, 80% power)                         ║
║  Model          moonshotai/kimi-k2-instruct-0905                   ║
║  L2 (refine)    enabled            L3 (plan)   enabled             ║
║  Scan context   YES                                                ║
║  Critique       enabled                                            ║
╚════════════════════════════════════════════════════════════════════╝


2026-03-24 13:45:46 INFO     [api.services.obs.observability_logger] Dataset 'termnorm_ground_truth': 728 items registered, 256 duplicates/empty skipped (from 984 input)
2026-03-24 13:45:46 WARNING  [api.services.obs.observability_logger] Skipping Langfuse cloud dataset registration for 984 items (rate-limit risk). Use the dedicated Langfuse sync cell instead.
2026-03-24 13:45:46 INFO     [api.services.campaign.feedback_cycle] Registered 728 dataset items for 'termnorm_ground_truth'
2026-03-24 13:45:46 INFO     [api.services.campaign.feedback_cycle] Registered prompt alias: b803a6fb ↔ 82ec89b9


TypeError: 'NoneType' object is not subscriptable

In [ ]:
#@title 5. Results — Campaign comparison, flip tracking, lineage
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

In [ ]:
#@title Save winner
save_campaign_winner(
    campaign_rounds, campaign_config, svc["store"], svc["backend_id"],
    experiment_id=locals().get("EXPERIMENT_ID"),
)

In [ ]:
#@title Generate LLM suggestions for next round
llm_client, llm_model = setup_llm(campaign_config)
suggestions = await generate_suggestions(
    campaign_rounds, eval_data, campaign_config,
    llm_client, model=llm_model,
)
display_suggestions(suggestions, len(campaign_rounds))
print("--- SUGGESTED CONFIG (copy to Setup) ---")
print(json.dumps(suggestions.get("suggested_config", campaign_config), indent=2))

In [ ]:
#@title Sync evaluation history to Langfuse
# Safe to re-run — already-pushed runs are skipped automatically.
stats = sync_langfuse(
    svc["store"], svc["backend_id"],
    dataset_name="termnorm_ground_truth",
)